# CRIVO — a melhor proporção entre as duas rodadas do v7c, escolhida no dev (sem treinar; T4 serve)

**De onde vem:** a média meio a meio das rodadas do v7c (`resumo_media_v7c.json`) passou o chat no blind v3 (+1,6, IC
[+0,6; +2,5]), mas perdeu em paráfrases (−2,0) e em urgência (115 contra 117). A rodada 2 (a do chat) é a mais forte em
paráfrases, então pesar mais a rodada 2 deve ajudar. Escolher a proporção olhando os blinds seria ajustar à régua; por
isso **a escolha é no dev**, e os blinds medem só a escolhida, **uma vez**.

**Fixado antes de rodar:**
1. **candidatas:** α (o peso da rodada 2) = 0,5, 0,6, 0,7, 0,8, 0,9 e 1,0 (1,0 é o próprio chat). Pesos = (1 − α)·rodada 1
   + α·rodada 2;
2. **dev de escolha:** o mesmo tipo de dev que escolheu os passos do v7c, inteiro: todas as frases novas de cena do dev da
   fase 5 v4 + todo o dev do v7c (`gerar_v7c.py`), julgado do mesmo jeito (sentido estrito, com o IDENTIDADE.md aceito);
3. **regra:** fica a de **maior sentido no dev**, desde que a **urgência no dev não fique abaixo da do chat** (α = 1,0).
   Empate: a de α maior (mais perto do chat). Se a escolhida for α = 1,0, o chat fica e os blinds não são medidos. Se for
   α = 0,5, os blinds já foram medidos (não passou) e não são medidos de novo;
4. **régua:** a de sempre contra o v7c do chat, nos mesmos blinds do v7c, uma medida (geração gulosa, pesos fixos).

Passar não liga nada: ligar é decisão do criador, depois da conversão para NumPy.

**Tempo:** sem treino. O dev com 6 candidatas leva uns 30 a 40 minutos na T4, e os blinds mais uns 30.

> **GPU (T4) → Executar tudo.** Saída: `CRIVO/transformer-crivo-46m-conversa-v7c-media-alfa/` (resumo: `resumo_media_alfa.json`).


In [ ]:
# Célula 0 — GPU, Drive, a base 6a (tokenizador) e as duas rodadas do v7c
import os, sys, json, shutil, subprocess, hashlib, time, math, random, glob, re, gzip
from pathlib import Path
from datetime import datetime
from collections import Counter
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np
import torch
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → A100, L4 ou T4.")
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| PyTorch", torch.__version__, flush=True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # T4 e mais antigas: float16 (bfloat16 lá é emulado, lento e sem atenção eficiente)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-crivo-46m" / "resumo_fase6a.json").is_file()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei CRIVO/transformer-crivo-46m/resumo_fase6a.json. Rode na conta onde a fase 6a treinou (hr87676620).")
R6A = json.loads((CRIVO / "transformer-crivo-46m" / "resumo_fase6a.json").read_text())
if not R6A.get("terminou") or not R6A.get("pesos_sha256"):
    raise SystemExit("ERRO: a fase 6a ainda não terminou. Termine a 6a primeiro ('Executar tudo' no notebook dela).")
BASE = Path(R6A["pasta"]) / "pesos.pt"
if not BASE.is_file():   # o Drive pode estar montado em outro caminho
    BASE = CRIVO / "transformer-crivo-46m" / Path(R6A["pasta"]).name / "pesos.pt"
if sha256(BASE) != R6A["pesos_sha256"]:
    raise SystemExit(f"ERRO: {BASE} não tem o SHA do resumo da 6a ({R6A['pesos_sha256'][:8]}…).")
BASE_SHA = R6A["pesos_sha256"]
PARTIDA = "6a"   # receita longa desde a base 6a, com substituição (o ajuste leve a partir do chat não melhorou no v6c)
USAR_PROFESSOR = True   # False: só as falas escritas pelo assistente
# O chat de hoje: o 46M v7c (rodada 2). É a régua: guarda no dev e linha de base pareada.
CHAT_SHA = "e4067e3b4c9705fde0a2063cc7a0da958ebad196ca193a672f75a1a10cbfda0a"
CHAT_PASTA = CRIVO / "transformer-crivo-46m-conversa-v7c"
CHAT_PESOS = next((p for p in sorted(CHAT_PASTA.glob("rodada*/conversa_*/pesos.pt")) if sha256(p) == CHAT_SHA), None)
if CHAT_PESOS is None:
    raise SystemExit("ERRO: não achei os pesos do v7c do chat (sha e4067e3b…) em CRIVO/transformer-crivo-46m-conversa-v7c.")
R1_SHA = "d68bb3c26a65db9d954b56e9605a8d91608b846f0fda6857c8efa7c613d106f6"   # v7c rodada 1 (resumo_v7c_rodada1.json)
R1_PESOS = next((p for p in sorted(CHAT_PASTA.glob("rodada*/conversa_*/pesos.pt")) if sha256(p) == R1_SHA), None)
if R1_PESOS is None:
    raise SystemExit("ERRO: não achei os pesos da rodada 1 do v7c (sha d68bb3c2…) em CRIVO/transformer-crivo-46m-conversa-v7c.")
SAIDA = CRIVO / "transformer-crivo-46m-conversa-v7c-media-alfa"
SAIDA.mkdir(parents=True, exist_ok=True)
DADOS_DRIVE = CRIVO / "dados-fase6b"
DADOS_DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/fase6b")
LOCAL.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
print("CRIVO:", CRIVO, "| base 6a:", BASE, BASE_SHA[:12], flush=True)


In [ ]:
# Célula 1 — código do CRIVO
REPO = Path("/content/CRIVO")
BRANCH = "claude/crise-e-rotulos-professor"   # depois do merge: "main"
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
for p in (str(REPO), str(REPO / "scripts"), str(REPO / "experimentos" / "fase6b_conversa_humana"),
          str(REPO / "experimentos" / "fase5_conversa_20261013"), str(REPO / "experimentos" / "v6_entendimento_20261019")):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase6, gerar_fase6b, extras_fase6b, cenas_v4
import baixar_fontes_linguagem
import importlib.util
_spec = importlib.util.spec_from_file_location("formato_fase5", REPO / "artefatos" / "conversa_v4f_pt" / "formato_fase5.py")
formato_fase5 = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(formato_fase5)
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(BASE.parent / "tokenizer.json"))
ESP = {t: TOK.token_to_id(t) for t in ("<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>")}
DOC, FIM = ESP["<documento>"], ESP["<fim>"]
QUEM = gerar_fase6b.resumo_identidade()
# Fase 5: dados v1–v3 do repositório (SHA conferido) e v4 gerada aqui, como no notebook da v4.
EXP5 = REPO / "experimentos" / "fase5_conversa_20261013"
M = {n: json.loads((EXP5 / f).read_text(encoding="utf-8")) for n, f in (("v1", "manifesto.json"), ("v2", "manifesto_v2.json"),
                                                                       ("v3", "manifesto_v3.json"), ("v4", "manifesto_v4.json"))}
for arq, esperado in (("dados_fase5.json.gz", M["v1"]["arquivo_gz_sha256"]), ("dados_fase5_v2.json.gz", M["v2"]["arquivo_gz_sha256"]),
                      ("dados_fase5_v3.json.gz", M["v3"]["arquivo_gz_sha256"]), ("sentido.py", M["v4"]["sentido_sha256"]),
                      ("cenas_v4.py", M["v4"]["cenas_v4_sha256"]), ("gerar_v4.py", M["v4"]["gerar_v4_sha256"])):
    if sha256(EXP5 / arq) != esperado:
        raise SystemExit(f"ERRO: {arq} difere do manifesto")
V4_SHA_REGISTRADO = M["v4"]["dados_sha256"]
subprocess.run([sys.executable, str(EXP5 / "gerar_v4.py")], cwd=EXP5, check=True)
V4_SHA = json.loads((EXP5 / "manifesto_v4.json").read_text(encoding="utf-8"))["dados_sha256"]
print("dados da fase 5 v4:", "conferidos" if V4_SHA == V4_SHA_REGISTRADO else f"AVISO: sha {V4_SHA[:12]} ≠ registrado {V4_SHA_REGISTRADO[:12]}", flush=True)
import sentido
sentido_ok = cenas_v4.registrar_no_sentido(sentido)
ACEITAS_IDENT = extras_fase6b.aceitas_na_fase5(sentido.norm)   # respostas do IDENTIDADE.md, certas nessas situações
F5 = {n: json.loads(gzip.open(EXP5 / f).read().decode("utf-8")) for n, f in (("v1", "dados_fase5.json.gz"), ("v2", "dados_fase5_v2.json.gz"),
                                                                             ("v3", "dados_fase5_v3.json.gz"), ("v4", "dados_fase5_v4.json.gz"))}
F_SENTIDO = {k: v["sentido"]["certo"] for k, v in json.loads((EXP5 / "resultado_v4_resumo.json").read_text())["variantes"]["F"]["testes"].items()
             if isinstance(v, dict) and "sentido" in v}
# Dados v6 (falas da pessoa escritas pelo assistente, com autorização do criador; não são humanas)
EXP6 = REPO / "experimentos" / "v6_entendimento_20261019"
import comparar_v6
import substituicao_v6c, formas_grok
# v7e: o "chat" da régua é o v7c (R4/P4 mantêm os nomes do notebook do v7c).
R4 = {"pesos_sha256": CHAT_SHA, "pasta": str(CHAT_PESOS.parent)}
P4 = CHAT_PESOS
print("partida:", PARTIDA, "| pesos do chat (v7c):", P4, flush=True)


def com_o_chat(f):
    """Roda f() com o 46M v7c no lugar do modelo (as funções de fala leem o nome global `modelo`)."""
    global modelo
    proprio = modelo
    salvo = torch.load(P4, map_location="cpu", weights_only=True)
    chat = lp.LinguagemProfunda(lp.Configuracao(**dict(salvo["config"], dropout=0.0))).to(dispositivo)
    chat.load_state_dict(salvo["modelo"]); del salvo
    modelo = chat
    try:
        return f()
    finally:
        modelo = proprio
        del chat
        torch.cuda.empty_cache()

# Sem treino: uma medida só (ver o texto no topo).
RODADA, SEED = "media_alfa", 20261020
SAIDA_R = SAIDA
print("commit:", COMMIT, "| quem:", QUEM, "| v4 F (chat) no sentido:", F_SENTIDO, flush=True)


In [ ]:
# Célula 1b — os dados do v7c, os mesmos do chat (para os blinds serem os mesmos frase por frase)
import professor_v7 as PV, detector_gatilhos as DG, gerar_v6 as G6, gerar_v6b as G6B, formas_v7
M7_REPO = json.loads((EXP6 / "manifesto_v7c.json").read_text(encoding="utf-8"))   # o manifesto sem professor, versionado
for arq in ("formas_v7", "formas_v7_a", "formas_v7_b", "formas_v7_c", "formas_v7_d", "formas_v7_e", "formas_v7_saude",
            "formas_v7_identidade", "formas_v7c", "coerencia_professor", "detector_gatilhos", "substituicao_v6c"):
    if sha256(EXP6 / f"{arq}.py") != M7_REPO[f"{arq}_sha256"]:
        raise SystemExit(f"ERRO: {arq}.py difere do manifesto_v7c.json")
PROF_DIR = CRIVO / "dados-professor-v7"
PROF_DIR.mkdir(parents=True, exist_ok=True)
PROF_ARQ = PROF_DIR / "falas_professor_v7.jsonl.gz"
if USAR_PROFESSOR and not PROF_ARQ.is_file():
    raise SystemExit("Falta o arquivo do professor. Rode antes notebooks/professor_v7_colab.ipynb (numa sessão própria), depois "
                     "Ambiente de execução → Desconectar e excluir ambiente, e rode este de novo. Para treinar sem o professor: "
                     "USAR_PROFESSOR = False na célula 0.")
if USAR_PROFESSOR:
    MP = json.loads((PROF_DIR / "manifesto_professor_v7.json").read_text())
    print(f"professor: {MP['professor']['modelo']} ({MP['professor']['licenca']}) | falas lidas {MP['falas_lidas']} | "
          f"aprovadas na conferência {MP['aprovadas_na_conferencia']} | fora na leitura {MP['fora_na_leitura']}", flush=True)
    rr = random.Random(3)
    with gzip.open(PROF_ARQ, "rt", encoding="utf-8") as f:
        _amostra = [json.loads(l) for l in f]
    for x in rr.sample(_amostra, min(30, len(_amostra))):
        print(f"   [{x['situacao']}] {x['fala']}", flush=True)
    del _amostra

# os dados v7c (determinísticos dado o arquivo do professor)
subprocess.run([sys.executable, str(EXP6 / "gerar_v7c.py")] + (["--professor", str(PROF_ARQ)] if USAR_PROFESSOR else []),
               cwd=EXP6, check=True)
M6 = json.loads((EXP6 / "manifesto_v7c.json").read_text(encoding="utf-8"))
V6 = json.loads(gzip.open(EXP6 / "dados_v7c.json.gz").read().decode("utf-8"))
print("dados v7c:", M6["tamanhos"], "| falas por origem:", M6["falas_por_origem"], "| filtros:", M6["relatorio"], flush=True)


In [ ]:
# Célula 2 — dados de conversa (uma vez; depois vêm do Drive) e tokens
DADOS = DADOS_DRIVE / "dados_fase6b.json.gz"
if not DADOS.is_file():
    OASST = LOCAL / "oasst2.messages.jsonl.gz"
    if not OASST.is_file():
        baixar_fontes_linguagem.baixar("oasst2", LOCAL)
    LEGENDAS = gerar_fase6b.LEGENDAS_URL
    subprocess.run([sys.executable, str(REPO / "experimentos/fase6b_conversa_humana/gerar_fase6b.py"), "--oasst", str(OASST),
                    "--legendas", LEGENDAS, "--saida", str(LOCAL / "dados")], cwd=REPO, check=True)
    shutil.copyfile(LOCAL / "dados" / "manifesto_fase6b.json", DADOS_DRIVE / "manifesto_fase6b.json")
    shutil.copyfile(LOCAL / "dados" / "dados_fase6b.json.gz", DADOS)
MANIFESTO_DADOS = json.loads((DADOS_DRIVE / "manifesto_fase6b.json").read_text())
with gzip.open(DADOS, "rb") as f:
    bruto = f.read()
assert hashlib.sha256(bruto).hexdigest() == MANIFESTO_DADOS["dados_sha256"], "dados_fase6b.json.gz corrompido"
DADOS_CONVERSA = json.loads(bruto); del bruto
REL_EXTRAS = extras_fase6b.extras(DADOS_CONVERSA, QUEM)   # identidade, lembranças e filtro do OASST2 (determinístico)

def de_fase5(c):
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(c["historico"])]
    return {"origem": "fase5", "ctx": {"quem": QUEM, **(c["contexto"] or {})},
            "turnos": t + [{"papel": "usuario", "texto": c["mensagem"]}, {"papel": "assistente", "texto": c["resposta"]}]}
REP_F5, REP_V6 = (1, 3) if PARTIDA == "v4" else (3, 3)
FASE5_TREINO = [c for c in F5["v4"]["treino"] if c["situacao"][5:] not in extras_fase6b.FASE5_SUBSTITUIDAS]
FASE5_FICA, V6_ENTRA, REL_SUBST = substituicao_v6c.substituir(FASE5_TREINO, V6["treino"], REP_F5, REP_V6, 20261020, formas_grok.chave,
                                                                sem_antigos_entram=True)   # identidade: não tem casos na fase 5
substituicao_v6c.conferir(REL_SUBST, REP_V6)   # o peso de cada situação não muda
# v7c: a identidade também não ganha peso. Ela não tem casos na fase 5 (a 6b treina pelo IDENTIDADE.md), então cada caso
# novo de identidade (×REP_V6) tira a mesma conta de exemplos do IDENTIDADE.md (×REP_IDENT) do grupo que cobre a situação.
REP_IDENT = 1 if PARTIDA == "v4" else 4
GRUPOS_IDENT = {"quem_e": ("quem",), "nome_crivo": ("quem",), "ser_humano": ("pessoa", "humano"), "robo": ("robo",), "sente": ("sentir",)}
_novos_ident = Counter(c["situacao"][5:] for c in V6_ENTRA if c["situacao"][5:] in GRUPOS_IDENT)
_rr_ident, _tirar_ident, REL_IDENT = random.Random(20261021), set(), {}
for _s, _n in sorted(_novos_ident.items()):
    _cands = [i for i, e in enumerate(DADOS_CONVERSA["treino"]) if e["origem"] == "identidade"
              and e["chave"].split(":")[1] in GRUPOS_IDENT[_s] and i not in _tirar_ident]
    _k = min(len(_cands) // 2, math.ceil(_n * REP_V6 / REP_IDENT))
    _tirar_ident.update(_rr_ident.sample(_cands, _k))
    REL_IDENT[_s] = {"casos_novos": _n, "exposicoes_novas": _n * REP_V6, "identidade_tirados": _k, "exposicoes_tiradas": _k * REP_IDENT}
DADOS_CONVERSA["treino"] = [e for i, e in enumerate(DADOS_CONVERSA["treino"]) if i not in _tirar_ident]
assert all(abs(r["exposicoes_novas"] - r["exposicoes_tiradas"]) < REP_IDENT for r in REL_IDENT.values()), REL_IDENT
print("identidade sem ganhar peso:", REL_IDENT, flush=True)
print(f"substituição: {len(V6_ENTRA)} casos novos (×{REP_V6}) no lugar de {len(FASE5_TREINO) - len(FASE5_FICA)} antigos (×{REP_F5});"
      f" situações limitadas: { {s: (r['novos'], r['novos_pedidos']) for s, r in REL_SUBST.items() if r['novos'] < r['novos_pedidos']} }", flush=True)
DADOS_CONVERSA["treino"] += [de_fase5(c) for c in FASE5_FICA]
DADOS_CONVERSA["val"] += [de_fase5(c) for c in F5["v4"]["dev"]]
def de_v6(c):
    e = de_fase5(c); e["origem"] = "v6"; return e
DADOS_CONVERSA["treino"] += [de_v6(c) for c in V6_ENTRA]
DADOS_CONVERSA["val"] += [de_v6(c) for c in V6["dev"]]
REL_EXTRAS["v6_treino"], REL_EXTRAS["v6_dev"] = len(V6_ENTRA), len(V6["dev"])
REL_EXTRAS["fase5_substituidos"] = len(FASE5_TREINO) - len(FASE5_FICA)
REL_EXTRAS["v7_casos_por_origem"] = dict(Counter(c.get("origem") for c in V6_ENTRA))
REL_EXTRAS["identidade_sem_ganhar_peso"] = REL_IDENT
REL_EXTRAS["fase5_treino"], REL_EXTRAS["fase5_dev"] = len(F5["v4"]["treino"]), len(F5["v4"]["dev"])
print("acrescentados:", REL_EXTRAS, flush=True)
print("dados:", json.dumps(MANIFESTO_DADOS["por_origem"], ensure_ascii=False), flush=True)
print("relatório:", json.dumps(MANIFESTO_DADOS["relatorio"], ensure_ascii=False), flush=True)

# tokens: cada texto distinto é codificado uma vez, em lote
_textos = set()
for v in DADOS_CONVERSA.values():
    for e in v:
        _textos.add(formato_fase6.limpar(formato_fase6.serializar_contexto(e.get("ctx"))))
        _textos.update(formato_fase6.limpar(t["texto"]) for t in e["turnos"])
_textos = sorted(_textos)
CACHE = dict(zip(_textos, (e.ids for e in TOK.encode_batch(_textos, add_special_tokens=False))))
del _textos
def codificar(texto):
    ids = CACHE.get(texto)
    return ids if ids is not None else TOK.encode(texto, add_special_tokens=False).ids

CTX = 1024
EX = {}   # (parte, origem) -> lista de (ids uint16, alvo bool)
nao_cabe = Counter()
for parte, v in DADOS_CONVERSA.items():
    for e in v:
        try:
            ids, alvo = formato_fase6.montar(codificar, ESP, e.get("ctx"), e["turnos"], contexto=CTX)
        except ValueError:
            nao_cabe[e["origem"]] += 1
            continue
        EX.setdefault((parte, e["origem"]), []).append((np.asarray(ids, np.uint16), np.asarray(alvo, bool)))
TOKENS_ALVO = {f"{p}/{o}": int(sum(a.sum() for _, a in l)) for (p, o), l in sorted(EX.items())}
print("exemplos:", {f"{p}/{o}": len(l) for (p, o), l in sorted(EX.items())}, "| não couberam:", dict(nao_cabe), flush=True)
print("tokens com perda:", TOKENS_ALVO, flush=True)



In [ ]:
# Célula 4 — as duas rodadas do v7c, e a mistura com peso α na rodada 2
_r1 = torch.load(R1_PESOS, map_location="cpu", weights_only=True)
_r2 = torch.load(CHAT_PESOS, map_location="cpu", weights_only=True)
assert _r1["config"] == _r2["config"] and _r1["modelo"].keys() == _r2["modelo"].keys()
R1_SD = {k: v.float() for k, v in _r1["modelo"].items()}
R2_SD = {k: v.float() for k, v in _r2["modelo"].items()}
TIPOS = {k: v.dtype for k, v in _r2["modelo"].items()}
CFG = dict(_r2["config"], dropout=0.0)
assert CFG["contexto"] >= CTX, CFG
del _r1, _r2
modelo = lp.LinguagemProfunda(lp.Configuracao(**CFG)).to(dispositivo)

def misturar(alfa):
    """Carrega no modelo (1 − α)·rodada 1 + α·rodada 2."""
    modelo.load_state_dict({k: ((1 - alfa) * R1_SD[k] + alfa * R2_SD[k]).to(TIPOS[k]) for k in R2_SD})
    modelo.eval()

_dist = sum((R1_SD[k] - R2_SD[k]).norm() ** 2 for k in R2_SD) ** 0.5
_tot = sum(R2_SD[k].norm() ** 2 for k in R2_SD) ** 0.5
print(f"rodadas carregadas: {sum(p.numel() for p in modelo.parameters())/1e6:.1f}M parâmetros | distância relativa {float(_dist / _tot):.3f}", flush=True)


In [ ]:
# Célula 5 — as funções de fala e de julgamento (as mesmas do treino)
MEMORIA_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
PROIBIDOS = [ESP["<pad>"], DOC, ESP["<usuario>"], ESP["<assistente>"]]
FREIO = {"penalidade": 1.3, "sem_repetir_ngrama": 3, "top_k": 40}
SEM_FREIO = {"penalidade": 1.0, "sem_repetir_ngrama": 10 ** 9, "top_k": 40}   # como a v4 F foi medida

def caso_ids(c):
    ctx = {"quem": QUEM, **(c["contexto"] or {})}
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(c["historico"])]
    ids, _ = formato_fase6.montar(codificar, ESP, ctx, t + [{"papel": "usuario", "texto": c["mensagem"]}], contexto=CTX - 120, gerar=True)
    return ids

@torch.no_grad()
def gerar_lote_freado(prompts, max_novos=110):
    # Gulosa, várias de uma vez (prompts do mesmo tamanho), com o mesmo freio de repetição da célula 6.
    modelo.eval()
    ids = torch.tensor(prompts, device=dispositivo)
    vistos = torch.zeros(len(prompts), CFG["vocabulario"], dtype=torch.bool, device=dispositivo)
    vivos, saidas, fim = list(range(len(prompts))), [[] for _ in prompts], [False] * len(prompts)
    k = FREIO["sem_repetir_ngrama"] - 1
    for _ in range(max_novos):
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(ids)[0][:, -1].float()
        lg[:, PROIBIDOS] = -float("inf")
        lg = torch.where(vistos, torch.where(lg > 0, lg / FREIO["penalidade"], lg * FREIO["penalidade"]), lg)
        for j, i in enumerate(vivos):
            s_ = saidas[i]
            if len(s_) >= k:
                proib = [s_[a + k] for a in range(len(s_) - k) if s_[a:a + k] == s_[-k:]]
                if proib:
                    lg[j, proib] = -float("inf")
        nxt = lg.argmax(-1).tolist()
        manter = []
        for j, (i, t) in enumerate(zip(vivos, nxt)):
            if t == FIM:
                fim[i] = True
            else:
                saidas[i].append(t); manter.append(j)
        if not manter:
            break
        idx = torch.tensor(manter, device=dispositivo)
        novos = torch.tensor([[nxt[j]] for j in manter], device=dispositivo)
        ids = torch.cat([ids[idx], novos], 1)
        vistos = vistos[idx]; vistos.scatter_(1, novos, True)
        vivos = [vivos[j] for j in manter]
    return saidas, fim

def avaliar_blind(casos, lote_max=None):
    lote_max = lote_max or (96 if MEMORIA_GB > 30 else 32)   # GPU menor: menos frases por vez
    prep = []
    for c in casos:
        try:
            prep.append((c, caso_ids(c)))
        except ValueError:
            pass
    grupos = {}
    for n, (_, p) in enumerate(prep):
        grupos.setdefault(len(p), []).append(n)
    textos, fins = [None] * len(prep), [False] * len(prep)
    for _, idx in sorted(grupos.items()):
        for a in range(0, len(idx), lote_max):
            parte = idx[a:a + lote_max]
            sa, fi = gerar_lote_freado([prep[n][1] for n in parte])
            for n, s_, f_ in zip(parte, sa, fi):
                textos[n], fins[n] = TOK.decode(s_).strip(), f_
    linhas = []
    for (c, _), texto, fim_ in zip(prep, textos, fins):
        av = formato_fase5.avaliar_saida(c, texto, fim_)
        extra = (c.get("exigir") or {}).get("contem_texto")
        if extra:
            av["contem_texto"] = extra.casefold() in texto.casefold()
            av["tudo_ok"] = av["tudo_ok"] and av["contem_texto"]
        linha = {"situacao": c["situacao"], "contraexemplo": bool(c.get("contraexemplo")), "mensagem": c["mensagem"],
                 "alvo": c["resposta"], "saida": texto, **av}
        if c.get("aceita"):
            linha["sentido"] = cenas_v4.julgar_real(sentido, c, texto)
            linha["aceita"], linha["historico"] = c["aceita"], c["historico"][-2:]
        else:
            j = sentido.julgar(linha)
            linha["sentido"] = None if j is None else j[0]
            if j is not None and j[0] != "certo" and sentido.norm(texto) in ACEITAS_IDENT.get(c["situacao"][5:], ()):
                linha["sentido_ident"] = "certo"   # resposta do IDENTIDADE.md nessa situação
            if j is not None and j[0] != "certo" and j[1] & comparar_v6.VIZINHAS.get(c["situacao"][5:], set()):
                linha["sentido_vizinha"] = "certo"   # vizinha da lista fixada antes (conta à parte)
        linha["chave"] = comparar_v6.chave(c)
        linhas.append(linha)
    return linhas

# Grupos da guarda do dev (v7c): situações que caíram no v7 ou no v7b, ou que decidem um critério
GUARDA = {"urgencia": {"saude_urgente", "ajuda_socorro", "ansioso_peito"},
          "identidade": {"quem_e", "nome_crivo", "ser_humano", "robo", "sente", "ser_ia", "confirma_crivo", "consciencia"},
          "conversa": {"reciproco", "estou_bem", "oi_tudo_bem", "tem_certeza", "nao_era_isso", "nao_entendeu_queixa",
                       "voce_confunde", "dificil_falar", "nao_sabe_entao", "quem_sou_eu"},
          "saude": {"saude_geral", "ansioso", "estou_mal", "cansado", "sono", "medo"}}

def grupos_dev(linhas):
    out = {}
    for g, sits in GUARDA.items():
        ls = [l for l in linhas if l["sentido"] is not None and l["situacao"][5:] in sits]
        out[g] = [sum(l["sentido"] == "certo" or l.get("sentido_ident") == "certo" for l in ls), len(ls)]
    return out

def certo_ident(linhas):
    # Sentido certo contando também as respostas do IDENTIDADE.md nas situações que ele cobre.
    cenas = [l for l in linhas if l["sentido"] is not None]
    return round(sum(l["sentido"] == "certo" or l.get("sentido_ident") == "certo" for l in cenas) / max(1, len(cenas)), 4)



def estratificar(casos, n, semente):
    por = {}
    for c in casos:
        por.setdefault(c["situacao"], []).append(c)
    rr = random.Random(semente)
    for v in por.values():
        rr.shuffle(v)
    out = []
    while len(out) < n and any(por.values()):
        for k_ in sorted(por):
            if por[k_] and len(out) < n:
                out.append(por[k_].pop())
    return out


In [ ]:
# Célula 5b — a escolha da proporção no dev (regra fixada no texto do topo)
ALFAS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
DEV_ESCOLHA = ([c for c in F5["v4"]["dev"] if c.get("frase_nova") and c["situacao"].startswith("cena:")]
               + estratificar(V6["dev"], len(V6["dev"]), 20261018))   # o dev do v7c inteiro
print("dev de escolha:", len(DEV_ESCOLHA), "frases", flush=True)
DEV_ARQ = SAIDA / "dev_por_alfa.json"
CHAVES_ESCOLHA = [comparar_v6.chave(c) for c in DEV_ESCOLHA]
DEV_POR_ALFA = {}
if DEV_ARQ.is_file():
    _d = json.loads(DEV_ARQ.read_text())
    if _d.get("chaves") == CHAVES_ESCOLHA:
        DEV_POR_ALFA = {float(k): v for k, v in _d["por_alfa"].items()}
for alfa in ALFAS:
    if alfa in DEV_POR_ALFA:
        continue
    t0 = time.time()
    misturar(alfa)
    ls = avaliar_blind(DEV_ESCOLHA)
    DEV_POR_ALFA[alfa] = {"sentido_dev": certo_ident(ls), "grupos_dev": grupos_dev(ls)}
    DEV_ARQ.write_text(json.dumps({"chaves": CHAVES_ESCOLHA, "por_alfa": {str(k): v for k, v in DEV_POR_ALFA.items()}}, ensure_ascii=False))
    print(f"α = {alfa:.1f}: sentido no dev {DEV_POR_ALFA[alfa]['sentido_dev']:.4f} | grupos {DEV_POR_ALFA[alfa]['grupos_dev']} | {time.time() - t0:.0f}s", flush=True)

URG_CHAT = DEV_POR_ALFA[1.0]["grupos_dev"]["urgencia"][0]
validas = [a for a in ALFAS if DEV_POR_ALFA[a]["grupos_dev"]["urgencia"][0] >= URG_CHAT]
ALFA = max(validas, key=lambda a: (DEV_POR_ALFA[a]["sentido_dev"], a))
print(f"\nESCOLHIDA no dev: α = {ALFA:.1f} (sentido {DEV_POR_ALFA[ALFA]['sentido_dev']:.4f}; chat {DEV_POR_ALFA[1.0]['sentido_dev']:.4f}; "
      f"urgência {DEV_POR_ALFA[ALFA]['grupos_dev']['urgencia']} contra {DEV_POR_ALFA[1.0]['grupos_dev']['urgencia']} do chat)", flush=True)
ESCOLHA = {"alfa": ALFA, "dev_por_alfa": {str(k): v for k, v in DEV_POR_ALFA.items()}, "regra": "maior sentido no dev com urgência ≥ a do chat; empate: α maior",
           "n_dev": len(DEV_ESCOLHA)}
(SAIDA / "escolha_alfa.json").write_text(json.dumps(ESCOLHA, ensure_ascii=False, indent=1))
if ALFA == 1.0:
    raise SystemExit("No dev, nenhuma mistura passou o chat: o v7c fica como está, e os blinds não são medidos. Avise o assistente.")
if ALFA == 0.5:
    raise SystemExit("No dev, a melhor é a meio a meio, que já foi medida nos blinds (não passou). Avise o assistente.")
misturar(ALFA)
PASSO_MELHOR = f"media_alfa_{ALFA:.1f}"


In [ ]:
# Célula 6 — a média: o que ela diz (sem retoque)
@torch.no_grad()
def gerar_freado(ids, max_tokens, temperatura, semente):
    # Como modelo.gerar, com freio de repetição: tokens já ditos perdem força e nenhum trecho de 3 tokens se repete.
    modelo.eval(); g = torch.Generator(device=dispositivo).manual_seed(semente)
    ids, saida = list(ids), []
    for _ in range(max_tokens):
        x = torch.tensor([ids[-CFG["contexto"]:]], dtype=torch.long, device=dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(x)[0][0, -1].float()
        lg[PROIBIDOS] = -float("inf")
        if saida:
            vistos = torch.tensor(sorted(set(saida)), device=dispositivo)
            v = lg[vistos]; lg[vistos] = torch.where(v > 0, v / FREIO["penalidade"], v * FREIO["penalidade"])
        k = FREIO["sem_repetir_ngrama"] - 1
        if len(saida) >= k:
            proib = [saida[i + k] for i in range(len(saida) - k) if saida[i:i + k] == saida[-k:]]
            if proib:
                lg[proib] = -float("inf")
        if temperatura == 0:
            prox = int(lg.argmax())
        else:
            lg = lg / temperatura
            lg[lg < lg.topk(FREIO["top_k"]).values[-1]] = -float("inf")
            prox = int(torch.multinomial(lg.softmax(-1), 1, generator=g))
        if prox == FIM:
            return saida, True
        saida.append(prox); ids.append(prox)
    return saida, False

def falar(ctx, turnos, temperatura=0.0, semente=0, max_tokens=120):
    ids, _ = formato_fase6.montar(codificar, ESP, ctx, turnos, contexto=CTX - max_tokens, gerar=True)
    saida, terminou = gerar_freado(ids, max_tokens, temperatura, semente)
    return TOK.decode(saida).strip(), terminou

def turnos_de(historico, mensagem):
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(historico)]
    return t + [{"papel": "usuario", "texto": mensagem}]

def conferir(ctx, historico, mensagem, texto, terminou, exigir=None):
    caso = {"contexto": {k: v for k, v in (ctx or {}).items() if k in formato_fase5.CAMPOS}, "historico": list(historico),
            "mensagem": mensagem + " " + " ".join((ctx or {}).get("lembrancas", [])) + " " + (ctx or {}).get("quem", ""),
            "exigir": exigir or {}}
    r = formato_fase5.avaliar_saida(caso, texto, terminou)
    return {k: v for k, v in r.items() if k in ("tudo_ok", "usa_fato", "inventados", "sem_repeticao", "terminou")}

FALAS = []
def ouvir(grupo, ctx, historico, mensagem, exigir=None, espera=None):
    for temp, sem in ((0.0, 0), (0.7, 1)):
        texto, terminou = falar(ctx, turnos_de(historico, mensagem), temp, sem)
        FALAS.append({"grupo": grupo, "temperatura": temp, "historico": list(historico), "mensagem": mensagem,
                      "lembrancas": (ctx or {}).get("lembrancas"), "fatos": (ctx or {}).get("fatos"), "resposta": texto,
                      "terminou": terminou, "conferencia": conferir(ctx, historico, mensagem, texto, terminou, exigir),
                      "lembrou": bool(re.search(r"(?<!\w)" + re.escape(espera.casefold()) + r"(?!\w)", texto.casefold())) if espera else None})

Q = {"quem": QUEM}
for c in cenas_v4.CONVERSAS_REAIS:
    ouvir("frases_reais_do_criador", Q, c["historico"], c["mensagem"])
for m in ("quem é você?", "qual é o seu nome?", "você é uma pessoa?", "o que você mais gosta de fazer?", "você tem sentimentos?"):
    ouvir("identidade_treinada", Q, [], m)        # formas que estão no treino
for m in extras_fase6b.PERGUNTAS_AVALIACAO + ["do que você é feito?"]:
    ouvir("identidade_nova", Q, [], m)            # formas que não estão no treino
for m in ("me dá uma dica pra estudar melhor", "como faço arroz?", "por que o céu é azul?", "o que é mais importante na vida?",
          "tô cansado do trabalho", "me conta uma curiosidade", "qual a diferença entre vírus e bactéria?", "você gosta de música?"):
    ouvir("perguntas_livres", Q, [], m)
# Valores, tipos e perguntas fora do treino (extras_fase6b.FATOS não tem papagaio, irmã, Juiz de Fora, Remo, filosofia...)
for lembr, msg, espera in ((["a pessoa disse: meu nome é Rosângela"], "você sabe como eu me chamo?", "Rosângela"),
                           (["a pessoa disse: tenho um papagaio chamado Louro"], "qual o nome do meu papagaio?", "Louro"),
                           (["a pessoa disse: eu moro em Juiz de Fora", "a pessoa disse: tenho 40 anos"], "em que cidade eu vivo?", "Juiz de Fora"),
                           (["a pessoa disse: semana que vem tenho prova de filosofia"], "minha prova é de quê?", "filosofia"),
                           (["a pessoa disse: eu torço pro Remo"], "lembra qual é o meu time?", "Remo"),
                           (["a pessoa disse: minha irmã se chama Lúcia", "a pessoa disse: ela tem 7 anos"], "quantos anos tem a minha irmã?", "7"),
                           (["a pessoa disse: meu nome é Tânia"], "qual é o meu time?", "não me contou"),
                           (["a pessoa disse: eu moro em Natal"], "qual é o nome do meu cachorro?", "não me contou")):
    ouvir("lembrancas", {"quem": QUEM, "lembrancas": lembr}, [], msg, espera=espera)
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "acervo" and sum(f["grupo"] == "fatos_do_acervo" for f in FALAS) < 40:
        ouvir("fatos_do_acervo", e["ctx"], [], e["turnos"][0]["texto"], exigir={"fato": True})
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "oasst2" and sum(f["grupo"] == "oasst2_separado" for f in FALAS) < 30             and e["turnos"][-2]["texto"] not in {f["mensagem"] for f in FALAS}:
        ts = e["turnos"][:-1]
        ouvir("oasst2_separado", e["ctx"], [t["texto"] for t in ts[:-1]], ts[-1]["texto"])

def taxa_de(grupo, chave):
    v = [f["conferencia"].get(chave) if chave != "lembrou" else f["lembrou"] for f in FALAS if f["grupo"] == grupo and f["temperatura"] == 0.0]
    v = [x for x in v if x is not None]
    return round(sum(v) / len(v), 3) if v else None
TAXAS = {"conferencia_tudo_ok": {g: taxa_de(g, "tudo_ok") for g in sorted({f["grupo"] for f in FALAS})},
         "usa_fato_acervo": taxa_de("fatos_do_acervo", "usa_fato"), "lembrou": taxa_de("lembrancas", "lembrou")}
for f in FALAS:
    if f["temperatura"] == 0.0 and f["grupo"] in ("frases_reais_do_criador", "identidade_treinada", "identidade_nova", "lembrancas", "perguntas_livres"):
        print(f"[{f['grupo']}] {f['mensagem']!r}\n   → {f['resposta']}", flush=True)
print(json.dumps(TAXAS, ensure_ascii=False, indent=1), flush=True)


In [ ]:
# Célula 6b — a régua da v4 F: sentido nos blinds e nas frases reais, com o contexto que o chat monta
BLINDS = {"v3": F5["v3"]["blind_v3"], "v1": F5["v1"]["blind"], "parafrases": F5["v2"]["blind_parafrase"],
          "grok": F5["v2"]["blind_grok"], "v4": F5["v4"]["blind_v4"], "v6": V6["blind_v6"]}
SENTIDO, LINHAS = {}, {}
for k, casos in BLINDS.items():
    t0 = time.time()
    ls = avaliar_blind(casos)
    a = sentido.analisar(ls, k)
    LINHAS[k] = comparar_v6.compacto(ls)
    SENTIDO[k] = {"certo": a["certo"], "certo_com_identidade": certo_ident(ls), "confundiu": a["confundiu"], "n": a["n_cenas"], "tudo_ok": formato_fase5.resumir(ls).get("tudo_ok"),
                  "piores": a["piores"][:6],
                  "exemplos": [{x: l[x] for x in ("situacao", "mensagem", "saida", "sentido")} for l in ls[::max(1, len(ls) // 12)]][:12]}
    SENTIDO[k]["ic95"] = comparar_v6.ic95([comparar_v6.certo(l) for l in LINHAS[k]])
    SENTIDO[k]["com_vizinhas"] = comparar_v6.ic95([comparar_v6.certo(l, True) for l in LINHAS[k]])["media"]
    f_ = F_SENTIDO.get(k)
    print(f"sentido {k}: {100 * (a['certo'] or 0):.1f}% (lista da F) | {100 * SENTIDO[k]['certo_com_identidade']:.1f}% (com o IDENTIDADE.md)"
          + (f" | v4 F, no chat: {100 * f_:.1f}%" if f_ is not None else "")
          + f" | {time.time() - t0:.0f}s", flush=True)
FREIO_ATUAL = FREIO
FREIO = SEM_FREIO
ls_sem = avaliar_blind(F5["v3"]["blind_v3"])
SEM_FREIO_V3 = {"certo": sentido.analisar(ls_sem, "v3")["certo"], "certo_com_identidade": certo_ident(ls_sem)}
ls_sem = avaliar_blind(F5["v4"]["conversas_reais"])
SEM_FREIO_REAIS = sum(l["sentido"] == "certo" for l in ls_sem)
FREIO = FREIO_ATUAL
print(f"sem freio: sentido v3 {100 * (SEM_FREIO_V3['certo'] or 0):.1f}% (com o IDENTIDADE.md {100 * SEM_FREIO_V3['certo_com_identidade']:.1f}%) | "
      f"frases reais {SEM_FREIO_REAIS}/15", flush=True)
ls = avaliar_blind(F5["v4"]["conversas_reais"])
REAIS = {"certo": sum(l["sentido"] == "certo" for l in ls), "n": len(ls),
         "linhas": [{x: l.get(x) for x in ("historico", "mensagem", "saida", "sentido", "aceita")} for l in ls]}
print(f"\nfrases reais do criador: {REAIS['certo']}/{REAIS['n']} certas (v4 F: 14/15)", flush=True)
for l in REAIS["linhas"]:
    print(f"  [{l['sentido']}] {l['mensagem']!r}\n     → {l['saida']}", flush=True)


# Linha de base pareada: o v7c do chat, nas mesmas frases (uma vez; depois vem do Drive). O nome do arquivo é o que
# comparar_v6.py lê; o conteúdo é o v7c (pesos_sha256 dentro). As linhas da rodada 2 do v7c são o próprio chat: reaproveitadas
# blind a blind quando as frases são as mesmas; o que faltar é medido agora com o 46M.
BASE_LINHAS = SAIDA / "linhas_v4_base.json"
if BASE_LINHAS.is_file() and json.loads(BASE_LINHAS.read_text()).get("pesos_sha256") != CHAT_SHA:
    raise SystemExit(f"ERRO: {BASE_LINHAS} não é do v7c do chat. Apague o arquivo para medir de novo.")
if not BASE_LINHAS.is_file():
    _v7c = CHAT_PASTA / "linhas_rodada2.json"
    _r = json.loads(_v7c.read_text()) if _v7c.is_file() else {"linhas": {}}
    base_linhas, medir_blinds = {}, {}
    for k, casos in BLINDS.items():
        chaves = [comparar_v6.chave(c) for c in casos]
        if [l["chave"] for l in _r["linhas"].get(k, [])] == chaves:
            base_linhas[k] = _r["linhas"][k]
        else:
            medir_blinds[k] = casos
    print("linha de base v7c: reaproveitados", sorted(base_linhas), "| medidos agora", sorted(medir_blinds), flush=True)
    base_linhas.update(com_o_chat(lambda: {k: comparar_v6.compacto(avaliar_blind(c)) for k, c in medir_blinds.items()}))
    reais_chat = _r.get("frases_reais")
    if reais_chat is None or medir_blinds:
        reais_chat = sum(l["sentido"] == "certo" for l in com_o_chat(lambda: avaliar_blind(F5["v4"]["conversas_reais"])))
    BASE_LINHAS.write_text(json.dumps({"pesos_sha256": CHAT_SHA, "modelo": "46M v7c (chat)", "linhas": base_linhas,
                                       "frases_reais": reais_chat}, ensure_ascii=False))
    print("linha de base (v7c do chat) guardada:", BASE_LINHAS, flush=True)
print(comparar_v6.tabela({f"α={ALFA:.1f}": LINHAS}), flush=True)
JULGAMENTO = None
if BASE_LINHAS.is_file():
    B = json.loads(BASE_LINHAS.read_text())
    print(comparar_v6.tabela({"v7c (chat)": B["linhas"]}), flush=True)
    JULGAMENTO = comparar_v6.julgar_rodada(LINHAS, B["linhas"], REAIS["certo"], TAXAS["usa_fato_acervo"], com_urgencia=True)
    for k, d in JULGAMENTO["diferencas"].items():
        print(f"  {k:11s} α={ALFA:.1f} − v7c = {100 * d['media']:+.1f} pontos [{100 * d['ic95'][0]:+.1f}; {100 * d['ic95'][1]:+.1f}]", flush=True)
    print(f"urgência (acertos α={ALFA:.1f} / v7c do chat):", JULGAMENTO["urgencia_acertos"], flush=True)
    print("critérios desta rodada:", JULGAMENTO["criterios"], "| passou:", JULGAMENTO["passou"], flush=True)
(SAIDA / "linhas_media_alfa.json").write_text(json.dumps({"linhas": LINHAS, "frases_reais": REAIS["certo"],
                                                              "usa_fato": TAXAS["usa_fato_acervo"], "seed": SEED}, ensure_ascii=False))


In [ ]:
# Célula 7 — salvar a média e o resumo pequeno
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA / f"conversa_{stamp}"
DEST.mkdir(parents=True)
torch.save({"versao": lp.VERSAO, "modelo": {k: v.detach().cpu() for k, v in modelo.state_dict().items()}, "config": CFG,
            "passo": PASSO_MELHOR, "execucao": {"tokenizer_sha256": sha256(BASE.parent / "tokenizer.json"), "pesos_externos": False,
                                                 "protocolo": "v7c_media_alfa", "media_de": {"rodada1": R1_SHA, "rodada2": CHAT_SHA, "alfa": ALFA}, "escolha": ESCOLHA,
                                                 "commit": COMMIT, "dados_v6_sha256": M6["dados_sha256"]}}, DEST / "pesos.pt")
shutil.copyfile(BASE.parent / "tokenizer.json", DEST / "tokenizer.json")
shutil.copyfile(REPO / "experimentos/fase6b_conversa_humana/formato_fase6.py", DEST / "formato_fase6.py")
resumo = {"protocolo": "v7c_media_alfa", "media_de": {"rodada1": R1_SHA, "rodada2": CHAT_SHA, "alfa": ALFA}, "escolha": ESCOLHA,
          "distancia_relativa_r1_r2": float(_dist / _tot), "regua_contra": {"modelo": "46M v7c (chat)", "pesos_sha256": CHAT_SHA},
          "julgamento": JULGAMENTO, "criterios_fixados": {**comparar_v6.CRITERIOS, **comparar_v6.CRITERIO_URGENCIA},
          "uma_medida": "geração gulosa e pesos fixos: medir de novo dá o mesmo resultado; sem treino, não há rodadas",
          "dados_v7c": {k: M6[k] for k in ("dados_sha256", "tamanhos", "falas_por_origem")}, "criado_em": datetime.now().isoformat(),
          "commit": COMMIT, "gpu": GPU, "config": CFG, "freio": FREIO, "sentido": SENTIDO, "frases_reais": REAIS,
          "sem_freio": {"v3": SEM_FREIO_V3, "frases_reais": SEM_FREIO_REAIS}, "f_sentido": F_SENTIDO, "taxas": TAXAS, "falas": FALAS,
          "pasta": str(DEST), "pesos_sha256": sha256(DEST / "pesos.pt"), "ativo_no_chat": False}
(SAIDA / "resumo_media_alfa.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=1, default=float))
print("SALVO EM", DEST, "| sha", resumo["pesos_sha256"], flush=True)
print("Pronto. Avise o assistente: ele lê o resumo_media_alfa.json no Drive.", flush=True)
